In [1]:
import numpy as np
import glob
import os
import re
import sys

# This finds the project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# --- THIS IS THE LINE YOU ARE MISSING ---
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
# --- ADD THAT LINE! ---

print(f"Project root added to path: {PROJECT_ROOT}")


Project root added to path: /Users/minglu/Documents/Uni/bistable_model_simulation


In [2]:
np.sqrt(2*1500*2*1e-7)

np.float64(0.02449489742783178)

In [3]:
from simulation.solvers.spatial_process import simul_initialize, simul_run
from simulation.solvers.rate_conversions import calculate_kappas


from pathlib import Path
import os

In [4]:
""" Main execution block containing all physics parameters. """
###### ================================== 1. parameter setting =====================================
L = 2. # cubic box length

diff_scale = 1500. 
DA = 1. 
DB = 1. 
DX = 1.  
DX2 = 1. 

##### There are 6 reactions but only 4 sigma values
##### because the reactions B <-> X involve no sigma value
sigmas = np.array((1., 1., 1., 1.)) * 0.1 # sigma_r1f, sigma_r1b, sigma_r2f, sigma_r2b

box_shape = np.array((L, L, L,))

##### the Part to change freely for the corresponding simulation
# Schloegl's model reaction rates
k = np.array((0.15, 0.025, 5.75, 25.))
print("Reaction rates for bistable schloegl's model: ",k)
# full model reaction rates
ls = np.array((1.5, 1500., 150., 25., 5.75, 25.))
# ls = np.array((3., 1500., 75., 12.5, 5.75, 25.))
print("Reaction rates for bistable full model: ",ls)
    

Reaction rates for bistable schloegl's model:  [ 0.15   0.025  5.75  25.   ]
Reaction rates for bistable full model:  [   1.5  1500.    150.     25.      5.75   25.  ]


In [5]:
print("The difference of propensity for second channel on macroscopic level:")
print(f"Low state: {ls[2]*10-ls[3]*60/8}")
print(f"High state: {ls[2]*10-ls[3]*280/8}")
print("The difference of propensity for third channel on macroscopic level:")
print(f"Low state: {ls[4]*20-ls[5]*60/8}")
print(f"High state: {ls[4]*20-ls[5]*280/8}")

The difference of propensity for second channel on macroscopic level:
Low state: 1312.5
High state: 625.0
The difference of propensity for third channel on macroscopic level:
Low state: -72.5
High state: -760.0


In [6]:
range = [0.2, 0.5, 1.0, 2.0, 5.0, 10.0]

In [7]:
for i in range:
    print(f"Current diffusion coeff is: {diff_scale/i}.")
    diffusions = np.array((DX, DX2, DA, DB)) * diff_scale / i
    print(f"Diffusions are : {diffusions}.")
    kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)
    print("The difference of propensity for second channel on microscopic level:")
    print(f"Low state: {kappas[2]*10-kappas[3]*60/8}")
    print(f"High state: {kappas[2]*10-kappas[3]*280/8}")

Current diffusion coeff is: 7500.0.
Diffusions are : [7500. 7500. 7500. 7500.].
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1633e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.6213e+04
κ₂⁻ = 6.0355e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.
The difference of propensity for second channel on microscopic level:
Low state: 316865.97637919366
High state: 150888.56018056843
Current diffusion coeff is: 3000.0.
Diffusions are : [3000. 3000. 3000. 3000.].
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1654e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.6835e+04
κ₂⁻ = 6.1392e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.
The difference of propensity for second channel on microscopic level:
Low state: 322310.2647123682
High state: 153481.078434461
Current diffusion coeff is: 1500.0.
Diffusions are : [1500. 150

In [8]:
diffusions = np.array((DX, DX2, DA, DB)) * 1500
kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)

✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1688e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.7921e+04
κ₂⁻ = 6.3201e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.


In [9]:
kappas

array([7.16881808e+02, 1.50000000e+03, 3.79207338e+04, 6.32012229e+03,
       5.75000000e+00, 2.50000000e+01])

In [10]:
kappas[2]/kappas[3]

np.float64(6.0)

## calculate the dimensionaless kappa on the paper

In [11]:
def calculate_k_from_l(l):
    keq = l[0]/l[1]
    k = np.array((keq*l[2], keq*l[3], l[4], l[5]))
    return k

def get_reaction_volume(sigma):
    """Calculates the volume of the reaction sphere."""
    return (4.0/3.0) * np.pi * (sigma**3)


# --- Formula definitions ---

def l2_formula_notcoupled(kappa_1_plus, D, sigma):
     # Calculate the term inside the tanh function
    sqrt_term = np.sqrt(kappa_1_plus / (2 * D))
    
    # Calculate the Left-Hand Side (LHS) of the equation
    # This is the expression for the effective rate l_1^+
    tanh_val = np.tanh(sigma * sqrt_term)
    lhs = 4 * np.pi * (D + D) * (sigma - (1 / sqrt_term) * tanh_val)
    return lhs


In [12]:

from scipy.optimize import root_scalar, root

In [13]:
l2_formula_notcoupled(kappas[3],1500, 0.1)

np.float64(26.252463490509463)

In [14]:
l2_formula_notcoupled(kappas[2],1500, 0.1)

np.float64(151.2016696968037)

In [15]:
151.2016696968/26.2524634905

5.759523092052503

In [16]:
print("The difference of propensity for third channel on microscopic level:")
print(f"Low state: {kappas[4]*20-kappas[5]*60/8}")
print(f"High state: {kappas[4]*20-kappas[5]*280/8}")

The difference of propensity for third channel on microscopic level:
Low state: -72.5
High state: -760.0


In [17]:
def l1_plus_formula(kappa_1_plus, D, sigma):
     # Calculate the term inside the tanh function
    sqrt_term = np.sqrt(kappa_1_plus / (2 * D))
    
    # Calculate the Left-Hand Side (LHS) of the equation
    # This is the expression for the effective rate l_1^+
    tanh_val = np.tanh(sigma * sqrt_term)
    lhs = 4 * np.pi * D * (sigma - (1 / sqrt_term) * tanh_val)
    return lhs

def calculate_l2_rates(kappa_2_plus, kappa_2_minus, DA, DX, DX2, sigma_3):
    if kappa_2_plus <= 0 or kappa_2_minus <= 0: 
        return np.inf, np.inf
    
    alpha_sq = kappa_2_plus / (DX2 + DA) + kappa_2_minus / (DX2 + DX)
    alpha = np.sqrt(alpha_sq)
    common_factor = 4 * np.pi * (1 / alpha_sq) * (sigma_3 - np.tanh(alpha * sigma_3) / alpha)
    l2_plus = kappa_2_plus * common_factor
    l2_minus = kappa_2_minus * common_factor

    return l2_plus, l2_minus

In [18]:
D = [20, 500, 750, 1000, 1250, 1500, 1600]
print(f"Kappas are {kappas}")
for k, _ in enumerate(D):
    print(f" ----- Current D is {D[k]} ----- ")
    print("Calculated l is:")
    l1p = l1_plus_formula(kappas[0], D[k], sigma=sigmas[0])
    l2p, l2m = calculate_l2_rates(kappas[2], kappas[3], D[k], D[k], D[k], sigmas[0])
    print(f"l1p:{l1p:.2f}, l2p:{l2p:.2f}, l2m:{l2m:.2f}")

Kappas are [7.16881808e+02 1.50000000e+03 3.79207338e+04 6.32012229e+03
 5.75000000e+00 2.50000000e+01]
 ----- Current D is 20 ----- 
Calculated l is:
l1p:1.40, l2p:30.16, l2m:5.03
 ----- Current D is 500 ----- 
Calculated l is:
l1p:1.50, l2p:135.00, l2m:22.50
 ----- Current D is 750 ----- 
Calculated l is:
l1p:1.50, l2p:142.10, l2m:23.68
 ----- Current D is 1000 ----- 
Calculated l is:
l1p:1.50, l2p:145.94, l2m:24.32
 ----- Current D is 1250 ----- 
Calculated l is:
l1p:1.50, l2p:148.35, l2m:24.72
 ----- Current D is 1500 ----- 
Calculated l is:
l1p:1.50, l2p:150.00, l2m:25.00
 ----- Current D is 1600 ----- 
Calculated l is:
l1p:1.50, l2p:150.52, l2m:25.09


In [19]:
1.499158/1500

0.0009994386666666666

In [20]:
1.486914/1500

0.0009912760000000001

In [21]:
(5.7774-5.7451)/5.7451

0.0056221823815077576

In [22]:
150.670141/26.225923

5.745084396076355

In [23]:
150.434131/26.038389

5.777397787551297

In [24]:
150.643951/26.220204

5.745338632758158

In [25]:
def decoupled_l2_formula(kappa_1_plus, D, sigma):
     # Calculate the term inside the tanh function
    sqrt_term = np.sqrt(kappa_1_plus / (2 * D))
    
    # Calculate the Left-Hand Side (LHS) of the equation
    # This is the expression for the effective rate l_1^+
    tanh_val = np.tanh(sigma * sqrt_term)
    lhs = 8 * np.pi * D * (sigma - (1 / sqrt_term) * tanh_val)
    return lhs

def find_kappa_2(kappa_1_plus, l1_plus_input, D, sigma):
    """
    Defines the self-consistency equation for kappa_1_plus.
    This function will be zero when the correct kappa_1_plus is found.
    """
    # Ensure kappa_1_plus is positive to avoid math errors
    if kappa_1_plus <= 0:
        return np.inf # Return a large number if the guess is non-physical

    lhs = decoupled_l2_formula(kappa_1_plus, D, sigma)
    # Calculate the Right-Hand Side (RHS) of the equation
    rhs = l1_plus_input
    return lhs -rhs


In [26]:
solution = root_scalar(
            find_kappa_2,
            bracket=[1e4, 10e4],
            args=(150., 1500, 0.1), 
        )
k2p_sol = solution.root
solution = root_scalar(
            find_kappa_2,
            bracket=[1e3, 10e3],
            args=(25., 1500, 0.1), 
        )
k2m_sol = solution.root

In [27]:
print(k2p_sol)
print(k2m_sol)

37604.267964107996
6016.181045771754


In [28]:
k2p_sol/k2m_sol

6.250521332056112